In [16]:
from langgraph.graph import StateGraph,START, END;
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
import operator
from langgraph.checkpoint.memory import MemorySaver, InMemorySaver
from dotenv import load_dotenv

In [17]:
load_dotenv()

llm = ChatOpenAI()

In [18]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [19]:
def generate_joke(state: JokeState):
    prompt = f'Generate a joke on the topic {state['topic']}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [20]:
def generate_explanation(state: JokeState):
    prompt = f'Write an explanation for a joke - {state['joke']}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [21]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [24]:
config1 = {'configurable': {'thread_id': '1'}}
workflow.invoke({'topic': 'pizza'}, config=config)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [25]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f118d49-bc49-61ec-8002-084598509c84'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-05T20:48:05.068021+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f118d49-bc45-6664-8001-63d4d7bf2970'}}, tasks=(PregelTask(id='1a83286b-7c40-baac-6294-e03e0c4c1ac2', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='RateLimitError("Error code: 429 - {\'error\': {\'message\': \'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.\', \'type\': \'insufficient_quota\', \'param\': None, \'code\': \'insufficient_quota\'}}")', interrupts=(), state=None, result=None),), interrupts=())

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f118d49-bc49-61ec-8002-084598509c84'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-05T20:48:05.068021+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f118d49-bc45-6664-8001-63d4d7bf2970'}}, tasks=(PregelTask(id='1a83286b-7c40-baac-6294-e03e0c4c1ac2', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='RateLimitError("Error code: 429 - {\'error\': {\'message\': \'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.\', \'type\': \'insufficient_quota\', \'param\': None, \'code\': \'insufficient_quota\'}}")', interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza'},

In [28]:
config2 = {'configurable': {'thread_id': '2'}}
workflow.invoke({'topic': 'pasta'}, config=config2)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}